# TechMind AI — Extracción y Perfilado de Datasets

Este notebook centraliza la **extracción desde Kaggle** de los datasets candidatos para entrenar el modelo de clasificación de contenido técnico de **TechMind AI** (Etapa 2 — Construcción del Dataset).

Para cada dataset se realiza:
1. Descarga vía `kagglehub` (adapter de Pandas).
2. Perfilado rápido: dimensiones, columnas, tipos, nulos y muestra de registros.
3. Notas de relevancia para el caso de uso (clasificación temática de texto técnico).

Al final hay una **tabla comparativa** para apoyar la decisión de qué dataset(s) usar.

> Requisito: tener configurado el archivo `kaggle.json` (credenciales de la API de Kaggle) o haber ejecutado `kagglehub.login()` previamente.

## 0. Setup

In [90]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets] pandas
import os
import time
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)

### 0.1 Descubrir los archivos reales de cada dataset

`kagglehub` necesita un `file_path` **no vacío** apuntando a un archivo concreto dentro del dataset (extensiones soportadas: `.csv`, `.tsv`, `.json`, `.parquet`, `.xlsx`, etc.). Como estos datasets traen varios archivos, primero los descargamos y listamos para saber qué nombre poner en cada `file_path` de las secciones siguientes.

La API de Kaggle a veces responde con errores transitorios (ej. `502 Bad Gateway`); por eso cada descarga reintenta unas pocas veces antes de fallar.

In [91]:
handles = [
    "fabiochiusano/medium-articles",
    "hsankesara/medium-articles",
    "kutayahin/stackoverflow-programming-questions-2020-2025",
]


def descargar_con_reintentos(handle, intentos=3, espera_seg=5):
    for intento in range(1, intentos + 1):
        try:
            return kagglehub.dataset_download(handle)
        except Exception as e:
            if intento == intentos:
                raise
            print(f"   (intento {intento} falló: {e}. Reintentando en {espera_seg}s...)")
            time.sleep(espera_seg)


# Guardamos la ruta local de cada dataset (ya descargado/extraído aquí) para
# leer los CSV directamente con pandas más abajo, sin volver a descargarlos
# con kagglehub.dataset_load (eso duplicaba la descarga y causaba
# DataCorruptionError por un resume corrupto en archivos grandes).
archivos_por_dataset = {}
rutas_locales = {}
for handle in handles:
    ruta_local = descargar_con_reintentos(handle)
    archivos = sorted(os.listdir(ruta_local))
    archivos_por_dataset[handle] = archivos
    rutas_locales[handle] = ruta_local
    print(f"{handle}: ({ruta_local})")
    for archivo in archivos:
        print(f"   - {archivo}")
    print()

fabiochiusano/medium-articles: (C:\Users\will8\.cache\kagglehub\datasets\fabiochiusano\medium-articles\versions\2)
   - medium_articles.csv

hsankesara/medium-articles: (C:\Users\will8\.cache\kagglehub\datasets\hsankesara\medium-articles\versions\1)
   - articles.csv

kutayahin/stackoverflow-programming-questions-2020-2025: (C:\Users\will8\.cache\kagglehub\datasets\kutayahin\stackoverflow-programming-questions-2020-2025\versions\1)
   - stackoverflow_combined.csv
   - stackoverflow_combined_info.json
   - stackoverflow_combined_statistics.json



In [92]:
def reparar_cache(handle):
    """Fuerza una redescarga completa del dataset (bypassa un caché local corrupto)."""
    ruta_local = kagglehub.dataset_download(handle, force_download=True)
    rutas_locales[handle] = ruta_local
    archivos = sorted(os.listdir(ruta_local))
    archivos_por_dataset[handle] = archivos
    print(f"{handle} reparado: ({ruta_local})")
    for archivo in archivos:
        print(f"   - {archivo}")
    return ruta_local


# Descomenta la línea del dataset que necesites reparar, por ejemplo:
# reparar_cache("stackoverflow/stacksample")

In [93]:
def perfilar(nombre, df):
    """Resumen rápido de un dataset recién cargado."""
    print(f"=== {nombre} ===")
    print("Shape:", df.shape)
    print("\nColumnas y tipos:")
    print(df.dtypes)
    print("\nNulos por columna:")
    print(df.isna().sum())
    print("\nPrimeros 5 registros:")
    display(df.head())
    return {
        "dataset": nombre,
        "filas": df.shape[0],
        "columnas": df.shape[1],
        "nombres_columnas": list(df.columns),
    }

resumenes = []

---
## 1. Medium Articles (fabiochiusano)
`fabiochiusano/medium-articles`

Artículos de Medium con título, texto y tags. Buena fuente de artículos técnicos largos con temática variada.

In [94]:
# Set the path to the file you'd like to load
file_path = "medium_articles.csv"

# Leemos directo del archivo ya descargado por la celda 0.1
df_medium_chiusano = pd.read_csv(
    os.path.join(rutas_locales["fabiochiusano/medium-articles"], file_path)
)

resumenes.append(perfilar("fabiochiusano/medium-articles", df_medium_chiusano))

=== fabiochiusano/medium-articles ===
Shape: (192368, 6)

Columnas y tipos:
title        object
text         object
url          object
authors      object
timestamp    object
tags         object
dtype: object

Nulos por columna:
title        5
text         0
url          0
authors      0
timestamp    2
tags         0
dtype: int64

Primeros 5 registros:


,title,text,url,authors,timestamp,tags
0,Mental Note Vol. 24,"Photo by Josh Riemer on Unsplash\n\nMerry Christmas and Happy Holidays, everyone!\n\nWe just wanted everyone to know...",https://medium.com/invisible-illness/mental-note-vol-24-969b6a42443f,['Ryan Fan'],2020-12-26 03:38:10.479000+00:00,"['Mental Health', 'Health', 'Psychology', 'Science', 'Neuroscience']"
1,Your Brain On Coronavirus,Your Brain On Coronavirus\n\nA guide to the curious and troubling impact of the pandemic and isolation\n\nPhoto by c...,https://medium.com/age-of-awareness/how-the-pandemic-affects-our-brain-and-mental-health-ae2ec0a9fc1d,['Simon Spichak'],2020-09-23 22:10:17.126000+00:00,"['Mental Health', 'Coronavirus', 'Science', 'Psychology', 'Neuroscience']"
2,Mind Your Nose,Mind Your Nose\n\nHow smell training can change your brain in six weeks — and why it matters.\n\nBy Ann-Sophie Barwi...,https://medium.com/neodotlife/mind-your-nose-f0b097d533bb,[],2020-10-10 20:17:37.132000+00:00,"['Biotechnology', 'Neuroscience', 'Brain', 'Wellness', 'Science']"
3,The 4 Purposes of Dreams,Passionate about the synergy between science and technology to provide better care. Check out my newsletter: science...,https://medium.com/science-for-real/the-4-purposes-of-dreams-fc6719090e75,['Eshan Samaranayake'],2020-12-21 16:05:19.524000+00:00,"['Health', 'Neuroscience', 'Mental Health', 'Psychology', 'Science']"
4,Surviving a Rod Through the Head,"You’ve heard of him, haven’t you? Phineas Gage. The railroad worker who survived an explosion that involved an iron ...",https://medium.com/live-your-life-on-purpose/surviving-a-rod-through-the-head-2e5d74db978,['Rishav Sinha'],2020-02-26 00:01:01.576000+00:00,"['Brain', 'Health', 'Development', 'Psychology', 'Science']"


---
## 2. Medium Articles (hsankesara)
`hsankesara/medium-articles`

Dataset alternativo de artículos de Medium, de menor tamaño. Sirve como comparación o complemento del dataset anterior.

In [95]:
# Set the path to the file you'd like to load
file_path = "articles.csv"

# Leemos directo del archivo ya descargado por la celda 0.1
df_medium_hsankesara = pd.read_csv(
    os.path.join(rutas_locales["hsankesara/medium-articles"], file_path)
)

resumenes.append(perfilar("hsankesara/medium-articles", df_medium_hsankesara))

=== hsankesara/medium-articles ===
Shape: (337, 6)

Columnas y tipos:
author          object
claps           object
reading_time     int64
link            object
title           object
text            object
dtype: object

Nulos por columna:
author          0
claps           0
reading_time    0
link            0
title           0
text            0
dtype: int64

Primeros 5 registros:


,author,claps,reading_time,link,title,text
0,Justin Lee,8.3K,11,https://medium.com/swlh/chatbots-were-the-next-big-thing-what-happened-5fc49dd6fa61?source=---------0----------------,Chatbots were the next big thing: what happened? – The Startup – Medium,"Oh, how the headlines blared:\nChatbots were The Next Big Thing.\nOur hopes were sky high. Bright-eyed and bushy-tai..."
1,Conor Dewey,1.4K,7,https://towardsdatascience.com/python-for-data-science-8-concepts-you-may-have-forgotten-i-did-825966908393?source=-...,Python for Data Science: 8 Concepts You May Have Forgotten,"If you’ve ever found yourself looking up the same question, concept, or syntax over and over again when programming,..."
2,William Koehrsen,2.8K,11,https://towardsdatascience.com/automated-feature-engineering-in-python-99baf11cc219?source=---------2----------------,Automated Feature Engineering in Python – Towards Data Science,Machine learning is increasingly moving from hand-designed models to automatically optimized pipelines using tools s...
3,Gant Laborde,1.3K,7,https://medium.freecodecamp.org/machine-learning-how-to-go-from-zero-to-hero-40e26f8aa6da?source=---------3---------...,Machine Learning: how to go from Zero to Hero – freeCodeCamp,"If your understanding of A.I. and Machine Learning is a big question mark, then this is the blog post for you. Here,..."
4,Emmanuel Ameisen,935,11,https://blog.insightdatascience.com/reinforcement-learning-from-scratch-819b65f074d8?source=---------4----------------,Reinforcement Learning from scratch – Insight Data,"Want to learn about applied Artificial Intelligence from leading practitioners in Silicon Valley, New York, or Toron..."


---
## 3. StackOverflow Programming Questions 2020-2025
`kutayahin/stackoverflow-programming-questions-2020-2025`

Preguntas recientes de Stack Overflow (2020-2025). Útil para tener contenido técnico actualizado y evitar sesgo hacia tecnologías antiguas del StackSample original.

In [96]:
# Set the path to the file you'd like to load
file_path = "stackoverflow_combined.csv"

# Leemos directo del archivo ya descargado por la celda 0.1
# (el dataset también trae stackoverflow_combined_info.json y
# stackoverflow_combined_statistics.json con metadatos, no se usan aquí)
df_so_recent = pd.read_csv(
    os.path.join(rutas_locales["kutayahin/stackoverflow-programming-questions-2020-2025"], file_path)
)

resumenes.append(perfilar("kutayahin/stackoverflow-programming-questions-2020-2025", df_so_recent))

=== kutayahin/stackoverflow-programming-questions-2020-2025 ===
Shape: (95636, 34)

Columnas y tipos:
question_id                      int64
title                           object
body                            object
tags                            object
tag_count                        int64
programming_language            object
categories                      object
creation_date                   object
creation_year                    int64
creation_month                   int64
creation_weekday                 int64
last_activity_date              object
view_count                       int64
score                            int64
answer_count                     int64
comment_count                    int64
favorite_count                   int64
is_answered                       bool
has_accepted_answer               bool
accepted_answer_score            int64
has_code                          bool
code_block_count                 int64
title_word_count                 int64
t

,question_id,title,body,tags,tag_count,programming_language,categories,creation_date,creation_year,creation_month,creation_weekday,last_activity_date,view_count,score,answer_count,comment_count,favorite_count,is_answered,has_accepted_answer,accepted_answer_score,has_code,code_block_count,title_word_count,title_char_count,body_word_count,body_char_count,difficulty_score,quality_score,owner_reputation,owner_badge_count,first_response_time_seconds,first_response_time_hours,top_answer_score,top_answer_body_length
0,79810291,AttributeError: 'NoneType' object has no attribute 'get' in request from AI Tools,I am trying to develop a tool that agent (CodeAgent) will call on user input (prompt) in Gradio. This tool will inte...,"python, artificial-intelligence, huggingface, gradio",4,python,"bug, api, backend",2025-11-05 15:49:00,2025,11,2,2025-11-05 15:57:09,18,-1,0,0,0,False,False,0,True,2,12,81,36,190,0.200,0.70,345,0,NaN,NaN,0,0
1,79810195,"contextlib tries to change a ""frozen"" exception",I got a strange error in my pytest after some Dependabot updates. In my case it's triggered by an non-existent MinIO...,"python, minio",2,python,"bug, testing",2025-11-05 14:23:06,2025,11,2,2025-11-05 14:25:06,29,0,0,0,0,False,False,0,True,3,7,47,113,599,0.200,0.70,529,0,NaN,NaN,0,0
2,79810063,Tkinter window shrinks after embedding Matplotlib canvas — how to prevent it?,I'm building a multi-frame Tkinter app for a school project (NEA dashboard). One of the frames displays a Matplotlib...,"python, matplotlib, tkinter",3,python,"api, backend",2025-11-05 12:42:01,2025,11,2,2025-11-05 12:42:01,27,2,0,0,0,False,False,0,True,11,12,77,143,812,0.201,0.70,21,0,NaN,NaN,0,0
3,79810057,"CatalystAppError: {'code': 'FATAL ERROR', 'message': 'Catalyst headers are empty'} when initializing Catalyst app",I’m working on a chatbot project using the Zoho Catalyst SDK. My goal is to use the Catalyst Cache service to store ...,"python, zoho, zohocatalyst",3,python,bug,2025-11-05 12:36:07,2025,11,2,2025-11-05 12:45:43,48,0,0,0,0,False,False,0,True,3,13,113,124,905,0.202,0.68,9,0,NaN,NaN,0,0
4,79810049,Sympy : Problem with simplifying a Sympy vector from user-defined transformation,"I am creating my own coordinate system using the vector module, and came across some strange behavior. First, the fo...","python, sympy",2,python,bug,2025-11-05 12:28:43,2025,11,2,2025-11-05 14:38:19,44,1,0,0,0,False,False,0,True,7,11,80,109,710,0.199,0.70,1620,0,NaN,NaN,0,0


## 4. techmind_dataset_v1.csv

In [97]:
df_etchmind = pd.read_csv("techmind_dataset_v1.csv")

In [98]:
df_etchmind.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36714 entries, 0 to 36713
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   titulo     36714 non-null  object 
 1   texto      36714 non-null  object 
 2   categoria  36714 non-null  object 
 3   fuente     36714 non-null  object 
 4   tags       36714 non-null  object 
 5   calidad    25574 non-null  float64
dtypes: float64(1), object(5)
memory usage: 1.7+ MB


In [99]:
resumenes.append(perfilar("techmind_dataset_v1.csv (subida manual)", df_etchmind))

=== techmind_dataset_v1.csv (subida manual) ===
Shape: (36714, 6)

Columnas y tipos:
titulo        object
texto         object
categoria     object
fuente        object
tags          object
calidad      float64
dtype: object

Nulos por columna:
titulo           0
texto            0
categoria        0
fuente           0
tags             0
calidad      11140
dtype: int64

Primeros 5 registros:


,titulo,texto,categoria,fuente,tags,calidad
0,How do you get state in a custom Extractor?,I'm trying to use Axum's custom extractor to implement JWT authentication. I can print the contents of the State wit...,Seguridad,stackoverflow,"rust, jwt, rust-axum",1.0
1,FROM WEB DEVELOPMENT TO BI- EP01,From Web Development To Business intelligence\n\nHello everyone i decide to write this series of stories about my ex...,Backend,medium,"['Obiee', 'Odi', 'Oracle', 'Java', 'Jee']",NaN
2,pybind11: how to add document the exported module?,"I have some c++ code, and I successfully export it to python module using pybind11. The code is: My question is: how...",Programación General,stackoverflow,"python, c++, pybind11",0.9
3,Reference elements in matrices in Julia while assigning values in other matrices,I have a vector X whose elements are zeros and ones. I want to create another vector Z of the same size as X where e...,Ciencia de Datos,stackoverflow,"matlab, julia",1.0
4,"The Falcor data model is a graph, and the GraphQL data model is a tree.",Falcor\n\nOut of the two Falcor was my first love. It’s existance was validation of an idea I had explored before. T...,Frontend,medium,"['API', 'JavaScript', 'Nodejs', 'React', 'Software Development']",NaN


---
## 5. Comparativa y decisión

Tabla resumen de dimensiones y columnas de todos los datasets cargados, para decidir cuál(es) usar en el pipeline de `TechMind AI`.

In [100]:
df_resumen = pd.DataFrame(resumenes)
df_resumen

,dataset,filas,columnas,nombres_columnas
0,fabiochiusano/medium-articles,192368,6,"[title, text, url, authors, timestamp, tags]"
1,hsankesara/medium-articles,337,6,"[author, claps, reading_time, link, title, text]"
2,kutayahin/stackoverflow-programming-questions-2020-2025,95636,34,"[question_id, title, body, tags, tag_count, programming_language, categories, creation_date, creation_year, creation..."
3,techmind_dataset_v1.csv (subida manual),36714,6,"[titulo, texto, categoria, fuente, tags, calidad]"


---
## 6. Selección y unificación de columnas

Con base en la comparativa anterior, seleccionamos y renombramos las columnas relevantes de cada dataset a un esquema común (`titulo`, `texto`, `categoria`, `palabras_clave`) para el pipeline de TechMind AI:

| Columna final       | fabiochiusano/medium-articles | stackoverflow (kutayahin) | techmind_dataset_v1 |
|---------------------|--------------------------------|----------------------------|------------------------|
| **Título**          | `title`                        | `title`                    | `titulo`               |
| **Texto**           | `text`                         | `body`                     | `texto`                |
| **Categoría**       | *(no disponible)*              | `categories`                | `categoria`            |
| **Palabras Clave**  | `tags`                          | `tags`                     | `tags`                 |

Notas:
- **`hsankesara/medium-articles`** no tiene ninguna columna de categoría/tags (solo `author, claps, reading_time, link, title, text`), por lo que queda **fuera de la unificación** y se trabaja aparte más abajo (solo título y texto disponibles).
- **`Nivel`** no se calcula todavía en esta sección: se agrega en la **Parte 8**, usando `dificultad_so` (el `difficulty_score` original de StackOverflow, que sí se conserva aquí) más una heurística por palabras clave para los otros datasets.
- Se agrega una columna `fuente` para identificar el dataset de origen de cada fila.

In [101]:
def seleccionar_columnas(df, mapa_columnas, fuente):
    """Selecciona y renombra columnas de un dataset al esquema comun (titulo, texto, categoria, palabras_clave)."""
    df_sel = df[list(mapa_columnas.keys())].rename(columns=mapa_columnas)
    df_sel["fuente"] = fuente
    return df_sel


df_fabiochiusano_sel = seleccionar_columnas(
    df_medium_chiusano,
    {"title": "titulo", "text": "texto", "tags": "palabras_clave"},
    "fabiochiusano/medium-articles",
)
df_fabiochiusano_sel["categoria"] = np.nan
df_fabiochiusano_sel["dificultad_so"] = np.nan

df_so_sel = seleccionar_columnas(
    df_so_recent,
    {"title": "titulo", "body": "texto", "categories": "categoria", "tags": "palabras_clave"},
    "kutayahin/stackoverflow-programming-questions-2020-2025",
)
# Se conserva el difficulty_score original de SO (unica fuente con una medida
# de dificultad real) para poder derivar "nivel" mas adelante (Parte 8).
df_so_sel["dificultad_so"] = df_so_recent["difficulty_score"]

df_techmind_sel = seleccionar_columnas(
    df_etchmind,
    {"titulo": "titulo", "texto": "texto", "categoria": "categoria", "tags": "palabras_clave"},
    "techmind_dataset_v1 (subida manual)",
)
df_techmind_sel["dificultad_so"] = np.nan

columnas_finales = ["titulo", "texto", "categoria", "palabras_clave", "fuente", "dificultad_so"]
df_unificado = pd.concat(
    [
        df_fabiochiusano_sel[columnas_finales],
        df_so_sel[columnas_finales],
        df_techmind_sel[columnas_finales],
    ],
    ignore_index=True,
)

In [102]:

df_unificado.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 324718 entries, 0 to 324717
Data columns (total 6 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   titulo          324713 non-null  object 
 1   texto           324696 non-null  object 
 2   categoria       108458 non-null  object 
 3   palabras_clave  324718 non-null  object 
 4   fuente          324718 non-null  object 
 5   dificultad_so   95636 non-null   float64
dtypes: float64(1), object(5)
memory usage: 14.9+ MB


In [103]:
df_unificado.head()

,titulo,texto,categoria,palabras_clave,fuente,dificultad_so
0,Mental Note Vol. 24,"Photo by Josh Riemer on Unsplash\n\nMerry Christmas and Happy Holidays, everyone!\n\nWe just wanted everyone to know...",NaN,"['Mental Health', 'Health', 'Psychology', 'Science', 'Neuroscience']",fabiochiusano/medium-articles,NaN
1,Your Brain On Coronavirus,Your Brain On Coronavirus\n\nA guide to the curious and troubling impact of the pandemic and isolation\n\nPhoto by c...,NaN,"['Mental Health', 'Coronavirus', 'Science', 'Psychology', 'Neuroscience']",fabiochiusano/medium-articles,NaN
2,Mind Your Nose,Mind Your Nose\n\nHow smell training can change your brain in six weeks — and why it matters.\n\nBy Ann-Sophie Barwi...,NaN,"['Biotechnology', 'Neuroscience', 'Brain', 'Wellness', 'Science']",fabiochiusano/medium-articles,NaN
3,The 4 Purposes of Dreams,Passionate about the synergy between science and technology to provide better care. Check out my newsletter: science...,NaN,"['Health', 'Neuroscience', 'Mental Health', 'Psychology', 'Science']",fabiochiusano/medium-articles,NaN
4,Surviving a Rod Through the Head,"You’ve heard of him, haven’t you? Phineas Gage. The railroad worker who survived an explosion that involved an iron ...",NaN,"['Brain', 'Health', 'Development', 'Psychology', 'Science']",fabiochiusano/medium-articles,NaN


In [104]:
df_unificado.tail()

,titulo,texto,categoria,palabras_clave,fuente,dificultad_so
324713,Cannot submit login form with Selenium (Python) using Headless Remote Driver,I am trying to login to a website using Selenium. The relevant form HTML is as follows: I am using the Selenium Pyth...,Frontend,"python, html, selenium-webdriver",techmind_dataset_v1 (subida manual),NaN
324714,Room with KMM: Unresolved reference 'instantiateImpl',"I'm trying to implement Room into my Kotlin Multiplatform project, and Im using the official docs at https://develop...",Mobile,"android, ios, kotlin, gradle, kotlin-multiplatform",techmind_dataset_v1 (subida manual),NaN
324715,Things to know — Amazon Aurora DB,Features that are worth exploring\n\nParallel query for Aurora MySQL\n\nAurora MySQL parallel query is an optimizati...,DevOps / Cloud,"['Relational Databases', 'Getting Started', 'Things To Know', 'Amazon Aurora', 'AWS']",techmind_dataset_v1 (subida manual),NaN
324716,Create Polars DataFrame with Flattened Json File,The problem that I have is trying to read in a flattened json file into a polars dataframe in Rust. Here is the Json...,Programación General,"rust, rust-polars",techmind_dataset_v1 (subida manual),NaN
324717,FGC Character select Layout in REACT,I’m a beginner at React and have designed a custom FGC (Fighting Game Community) layout in Figma. I want to implemen...,Frontend,"javascript, css, reactjs, layout, css-grid",techmind_dataset_v1 (subida manual),NaN


---
## 7. Clasificación de `categoria` por palabras clave

`techmind_dataset_v1` define la taxonomía objetivo (8 categorías): `Backend`, `Programación General`, `Ciencia de Datos`, `Frontend`, `Mobile`, `DevOps / Cloud`, `Bases de Datos`, `Seguridad`.

Ni `fabiochiusano` (categoría faltante) ni `kutayahin/stackoverflow` (su `categories` original es tipo-de-pregunta: `bug`, `how-to`, `testing`, `performance`... no es temática) están en esa taxonomía. Para ambos se reclasifica `categoria` buscando, dentro de `palabras_clave`, coincidencias de palabras clave asociadas a cada categoría; se elige la categoría con más coincidencias.

Cuando ninguna palabra clave técnica coincide, el tratamiento difiere según el dataset:
- **StackOverflow** es intrínsecamente técnico → se asume `Programación General`.
- **fabiochiusano** incluye artículos no técnicos (salud, psicología, etc., ~84% de sus filas no matchean ninguna keyword) → esas filas se **descartan** de `df_unificado` en vez de forzarlas a `Programación General`, para no contaminar esa categoría con contenido no técnico.

`techmind_dataset_v1` **no se toca**: ya viene con la taxonomía correcta y es la fuente de verdad.

In [105]:
categorias_keywords = {
    "Frontend": [
        "html", "css", "javascript", "typescript", "react", "vue", "angular",
        "svelte", "frontend", "front-end", "dom", "jquery", "sass", "less",
        "tailwind", "webpack", "next.js", "nextjs", "bootstrap",
    ],
    "Backend": [
        "backend", "back-end", "api", "rest", "graphql", "node", "nodejs",
        "express", "django", "flask", "spring", "laravel", "microservice",
        "server-side", ".net", "asp.net", "fastapi", "php", "ruby-on-rails",
    ],
    "Mobile": [
        "android", "ios", "swift", "kotlin", "flutter", "react-native",
        "xamarin", "mobile", "dart", "objective-c", "swiftui", "jetpack",
    ],
    "Bases de Datos": [
        "sql", "mysql", "postgres", "postgresql", "mongodb", "database",
        "nosql", "redis", "sqlite", "oracle", "elasticsearch", "cassandra",
        "mariadb",
    ],
    "DevOps / Cloud": [
        "docker", "kubernetes", "k8s", "aws", "azure", "gcp", "google-cloud",
        "devops", "ci/cd", "cicd", "terraform", "jenkins", "cloud", "ansible",
        "nginx", "linux", "deployment",
    ],
    "Ciencia de Datos": [
        "machine-learning", "deep-learning", "data-science", "pandas",
        "numpy", "tensorflow", "pytorch", "nlp", "scikit-learn", "statistics",
        "data-analysis", "artificial-intelligence", "neural-network",
        "big-data", "matplotlib", "jupyter", "llm", "chatgpt",
    ],
    "Seguridad": [
        "security", "cybersecurity", "encryption", "authentication", "oauth",
        "jwt", "ssl", "tls", "vulnerability", "xss", "sql-injection",
        "penetration-testing", "firewall", "hashing",
    ],
    "Programación General": [
        "algorithm", "algorithms", "programming", "coding", "data-structures",
        "oop", "design-patterns", "git", "debugging", "c++", "c#", "java",
        "python", "go", "rust", "functional-programming",
    ],
}


def clasificar_categoria(palabras_clave):
    """Devuelve la categoria de techmind con mas coincidencias en palabras_clave, o None si ninguna aplica."""
    if pd.isna(palabras_clave):
        return None
    texto = str(palabras_clave).lower()
    conteos = {
        categoria: sum(1 for kw in keywords if kw in texto)
        for categoria, keywords in categorias_keywords.items()
    }
    mejor_categoria = max(conteos, key=conteos.get)
    return mejor_categoria if conteos[mejor_categoria] > 0 else None


mask_reclasificar = df_unificado["fuente"] != "techmind_dataset_v1 (subida manual)"
df_unificado.loc[mask_reclasificar, "categoria"] = df_unificado.loc[
    mask_reclasificar, "palabras_clave"
].apply(clasificar_categoria)

# StackOverflow es intrinsecamente tecnico: si no matchea ninguna keyword
# especifica, se asume "Programación General" en vez de descartar la fila.
mask_so_sin_categoria = (
    df_unificado["fuente"] == "kutayahin/stackoverflow-programming-questions-2020-2025"
) & df_unificado["categoria"].isna()
df_unificado.loc[mask_so_sin_categoria, "categoria"] = "Programación General"

# fabiochiusano NO es un dataset exclusivamente tecnico (incluye salud,
# psicologia, etc.): las filas sin ninguna coincidencia de keyword tecnica
# probablemente no son de programacion, asi que se descartan de df_unificado
# en vez de forzarlas a "Programación General".
filas_antes = len(df_unificado)
df_unificado = df_unificado[df_unificado["categoria"].notna()].reset_index(drop=True)
print(f"Filas descartadas (fabiochiusano sin categoria tecnica identificable): {filas_antes - len(df_unificado)}")

df_unificado["categoria"].value_counts()

Filas descartadas (fabiochiusano sin categoria tecnica identificable): 143076


categoria
Programación General    67347
Frontend                33350
Backend                 23035
Mobile                  19151
Bases de Datos          12897
DevOps / Cloud          11460
Ciencia de Datos         8777
Seguridad                5625
Name: count, dtype: int64

In [106]:
df_unificado["categoria"].value_counts()

categoria
Programación General    67347
Frontend                33350
Backend                 23035
Mobile                  19151
Bases de Datos          12897
DevOps / Cloud          11460
Ciencia de Datos         8777
Seguridad                5625
Name: count, dtype: int64

---
## 8. Clasificación de `nivel` (Básico / Intermedio / Avanzado)

`nivel` se calcula con dos métodos distintos según la fuente, porque solo StackOverflow trae una medida de dificultad real:

- **StackOverflow**: se usa `dificultad_so` (su `difficulty_score` original, 0-1). **No se usan terciles**: al revisar la distribución real, ~90% de las filas caen entre 0.20 y 0.21 (percentil 33 = 0.202, percentil 67 = 0.206), así que un split por cuantiles separaría Básico/Avanzado por una diferencia de ~0.004 — puro ruido, no dificultad real. En su lugar se usan **umbrales fijos** que respetan la forma real de la distribución: `< 0.15` → `Básico` (cola baja, ~2% de las filas), `> 0.22` → `Avanzado` (~9% superior, hasta 0.993), el resto (la mayoría, contenido "normal") → `Intermedio`.
- **fabiochiusano y techmind**: no tienen ningún campo de dificultad, así que se usa una **heurística por palabras clave** sobre `titulo` + `palabras_clave` (igual enfoque que en la Parte 7): términos como `beginner`, `introduction`, `basics` → `Básico`; términos como `advanced`, `architecture`, `internals`, `optimization` → `Avanzado`; si no hay ninguna coincidencia → `Intermedio` por defecto.

Se mezclan dos metodologías distintas en la misma columna (decisión tomada explícitamente): se aprovecha el dato real donde existe y se rellena con la heurística donde no.

In [107]:
nivel_keywords = {
    "Básico": [
        "beginner", "beginners", "introduction", "intro", "basics", "basic",
        "getting started", "getting-started", "fundamentals", "101",
        "first steps", "para principiantes", "primeros pasos", "easy",
        "simple", "tutorial",
    ],
    "Avanzado": [
        "advanced", "deep dive", "deep-dive", "internals", "architecture",
        "optimization", "performance tuning", "scalability", "distributed",
        "concurrency", "expert", "mastering", "in-depth", "under the hood",
        "avanzado", "arquitectura",
    ],
}


def clasificar_nivel_heuristico(titulo, palabras_clave):
    """Nivel basado en palabras clave de titulo/palabras_clave; 'Intermedio' si ninguna aplica."""
    partes = [str(v) for v in (titulo, palabras_clave) if pd.notna(v)]
    texto = " ".join(partes).lower()
    conteos = {
        nivel: sum(1 for kw in keywords if kw in texto)
        for nivel, keywords in nivel_keywords.items()
    }
    mejor_nivel = max(conteos, key=conteos.get)
    return mejor_nivel if conteos[mejor_nivel] > 0 else "Intermedio"


# StackOverflow: dificultad_so esta muy concentrada (~90% de las filas caen
# entre 0.20 y 0.21), asi que un split por terciles (pd.qcut) separaria
# Basico/Avanzado por una diferencia de ~0.004, puro ruido. Se usan umbrales
# fijos que respetan la forma real de la distribucion: la mayoria del
# contenido es "normal" (Intermedio), una cola baja es claramente mas facil
# (Basico) y solo el ~9% superior (score > 0.22, hasta 0.993) es Avanzado.
UMBRAL_BASICO_SO = 0.15
UMBRAL_AVANZADO_SO = 0.22

mask_so = df_unificado["fuente"] == "kutayahin/stackoverflow-programming-questions-2020-2025"
condiciones_so = [
    df_unificado.loc[mask_so, "dificultad_so"] < UMBRAL_BASICO_SO,
    df_unificado.loc[mask_so, "dificultad_so"] > UMBRAL_AVANZADO_SO,
]
df_unificado.loc[mask_so, "nivel"] = np.select(
    condiciones_so, ["Básico", "Avanzado"], default="Intermedio"
)

# fabiochiusano y techmind: sin difficulty_score, se usa la heurística por palabras clave.
mask_heuristico = ~mask_so
df_unificado.loc[mask_heuristico, "nivel"] = df_unificado.loc[mask_heuristico].apply(
    lambda fila: clasificar_nivel_heuristico(fila["titulo"], fila["palabras_clave"]), axis=1
)

df_unificado = df_unificado.drop(columns="dificultad_so")
df_unificado["nivel"].value_counts()

nivel
Intermedio    164081
Avanzado       10914
Básico          6647
Name: count, dtype: int64

In [108]:
df_unificado.to_csv("df_unificado.csv", index=False, encoding="utf-8")

---
## 9. Traducción de `titulo` y `palabras_clave` a español

Se traduce `df_unificado` a un nuevo `df_unificado_esp` usando `deep-translator` (Google Translate, gratis, sin API key). Alcance decidido:

- Se traducen **`titulo`** y **`palabras_clave`** para las ~163k filas completas.
- **`texto` se deja en inglés por ahora**: traducirlo también multiplicaría el volumen de llamadas y el tiempo de ejecución no sería viable con el motor gratuito (se puede retomar más adelante con otro motor si hace falta).

**Costo real medido**: una llamada individual tarda ~2.2s; con 20 hilos en paralelo el throughput efectivo baja a ~0.125s/llamada (probado con 300 llamadas, 0 errores). Con ~163,590 filas × 2 columnas, la estimación es de **~11 horas** de ejecución continua.

Por ser un proceso largo y dependiente de un servicio externo (riesgo de rate-limit/bloqueo de Google a mitad de camino), el proceso guarda un **checkpoint en CSV** después de cada lote: si se corta, basta con volver a correr la celda y continúa donde se quedó, en vez de reiniciar desde cero.

In [84]:
#!pip install deep-translator tqdm

In [88]:
'''import os
import time
from concurrent.futures import ThreadPoolExecutor, as_completed 
from deep_translator import GoogleTranslator
from tqdm.auto import tqdm

RUTA_CHECKPOINT = "df_unificado_esp_checkpoint.csv"
TAMANO_LOTE = 2000
MAX_HILOS = 20


def traducir_es(texto, intentos=3):
    """Traduce un texto de ingles a español, con reintentos ante fallos transitorios."""
    if pd.isna(texto) or str(texto).strip() == "":
        return texto
    for intento in range(1, intentos + 1):
        try:
            return GoogleTranslator(source="en", target="es").translate(str(texto))
        except Exception:
            if intento == intentos:
                return texto  # si falla definitivamente, se deja el original en inglés
            time.sleep(2 * intento)


def traducir_columna(serie, max_hilos=MAX_HILOS, descripcion=""):
    resultados = [None] * len(serie)
    with ThreadPoolExecutor(max_workers=max_hilos) as ex:
        futuros = {ex.submit(traducir_es, valor): i for i, valor in enumerate(serie)}
        for futuro in tqdm(
            as_completed(futuros), total=len(futuros), desc=descripcion, leave=False
        ):
            i = futuros[futuro]
            resultados[i] = futuro.result()
    return resultados


if os.path.exists(RUTA_CHECKPOINT):
    df_unificado_esp = pd.read_csv(RUTA_CHECKPOINT)
    fila_inicio = int(df_unificado_esp["traducido"].sum())
    print(f"Reanudando desde el checkpoint: {fila_inicio} filas ya traducidas.")
else:
    df_unificado_esp = df_unificado.copy()
    df_unificado_esp["traducido"] = False
    fila_inicio = 0

total_filas = len(df_unificado_esp)
lotes = list(range(fila_inicio, total_filas, TAMANO_LOTE))

for inicio in tqdm(lotes, desc="Lotes", unit="lote"):
    fin = min(inicio + TAMANO_LOTE, total_filas)
    df_unificado_esp.loc[inicio:fin - 1, "titulo"] = traducir_columna(
        df_unificado_esp.loc[inicio:fin - 1, "titulo"], descripcion=f"titulo [{inicio}:{fin}]"
    )
    df_unificado_esp.loc[inicio:fin - 1, "palabras_clave"] = traducir_columna(
        df_unificado_esp.loc[inicio:fin - 1, "palabras_clave"], descripcion=f"palabras_clave [{inicio}:{fin}]"
    )
    df_unificado_esp.loc[inicio:fin - 1, "traducido"] = True
    df_unificado_esp.to_csv(RUTA_CHECKPOINT, index=False)
    print(f"Progreso: {fin}/{total_filas} filas traducidas ({fin/total_filas:.1%})")

df_unificado_esp = df_unificado_esp.drop(columns="traducido")
print("Traducción completa.")
df_unificado_esp.head()
'''

'import os\nimport time\nfrom concurrent.futures import ThreadPoolExecutor, as_completed \nfrom deep_translator import GoogleTranslator\nfrom tqdm.auto import tqdm\n\nRUTA_CHECKPOINT = "df_unificado_esp_checkpoint.csv"\nTAMANO_LOTE = 2000\nMAX_HILOS = 20\n\n\ndef traducir_es(texto, intentos=3):\n    """Traduce un texto de ingles a español, con reintentos ante fallos transitorios."""\n    if pd.isna(texto) or str(texto).strip() == "":\n        return texto\n    for intento in range(1, intentos + 1):\n        try:\n            return GoogleTranslator(source="en", target="es").translate(str(texto))\n        except Exception:\n            if intento == intentos:\n                return texto  # si falla definitivamente, se deja el original en inglés\n            time.sleep(2 * intento)\n\n\ndef traducir_columna(serie, max_hilos=MAX_HILOS, descripcion=""):\n    resultados = [None] * len(serie)\n    with ThreadPoolExecutor(max_workers=max_hilos) as ex:\n        futuros = {ex.submit(traducir_

In [87]:
'''f_unificado.head()'''

'f_unificado.head()'

### Criterios sugeridos para elegir el/los dataset(s) final(es)

- **Relevancia temática**: ¿el contenido es técnico (código, documentación, tutoriales) o genérico?
- **Calidad de las etiquetas**: ¿existen categorías/tags claros y consistentes para usar como target?
- **Volumen y balance de clases**: suficientes ejemplos por categoría, sin desbalance extremo.
- **Actualidad**: contenido reciente evita sesgo hacia tecnologías obsoletas.
- **Longitud y limpieza del texto**: impacta directamente en el preprocesamiento (Etapa 3).

Una vez decidido, guardar el subconjunto final (crudo) en `dataset/raw/` y, tras la limpieza (Etapa 3), el resultado en `dataset/processed/`.